# Forecast Solar Mensual — FERCHEGAS El Viejón

**Método:** Perfil histórico calibrado por (mes, hora) + análogo histórico

- **Opción 3:** Promedio real de generación por (mes, hora) de los 3 años de histórico, escalado por rendimiento reciente. Captura estacionalidad solar Y variabilidad climática real.
- **Opción 1:** Generación del mismo período del año anterior, escalada por ratio de rendimiento.
- **Combinado:** 60% perfil histórico + 40% análogo

## 1. Carga de Datos

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_html('Ferchegas_El_viejon_3años.xls')[0]
df.columns = ['Datetime_str', 'Generacion_kWh', 'Pronostico_kWh']
df['Generacion_kWh'] = pd.to_numeric(df['Generacion_kWh'], errors='coerce').fillna(0.0)
df['Datetime'] = pd.to_datetime(df['Datetime_str'], errors='coerce')
df = df.dropna(subset=['Datetime']).sort_values('Datetime').reset_index(drop=True)
df['Hora'] = df['Datetime'].dt.hour
df['Mes']  = df['Datetime'].dt.month
df['Anio'] = df['Datetime'].dt.year
df['Dia']  = df['Datetime'].dt.date

print(f'Filas: {len(df)} | {df["Datetime"].min().date()} → {df["Datetime"].max().date()}')
print(f'Años con datos completos: {sorted(df["Anio"].unique())}')
print(f'Generación máxima horaria: {df["Generacion_kWh"].max():.1f} kWh')
print(f'Total generado 2024: {df[df["Anio"]==2024]["Generacion_kWh"].sum():,.0f} kWh')
print(f'Total generado 2025: {df[df["Anio"]==2025]["Generacion_kWh"].sum():,.0f} kWh')


Filas: 20808 | 2024-01-01 → 2026-05-17
Años con datos completos: [np.int32(2024), np.int32(2025), np.int32(2026)]
Generación máxima horaria: 85.4 kWh
Total generado 2024: 10,571 kWh
Total generado 2025: 49,499 kWh


## 2. Perfil Histórico por (Mes, Hora) — Opción 3

In [3]:
# Perfil: mediana de generación real por (mes, hora) — robusta ante outliers
# Solo usamos 2025 (primer año completo de operación; 2024 fue año de arranque con ~21% de gen)
df_hist = df[df['Anio'] == 2025]
perfil = df_hist.groupby(['Mes', 'Hora'])['Generacion_kWh'].median().reset_index()
perfil.columns = ['Mes', 'Hora', 'Gen_perfil']

print('Perfil de generación (mediana kWh) por mes y hora — solo 2025:')
pivot = perfil.pivot(index='Hora', columns='Mes', values='Gen_perfil').round(2)
pivot.columns = [f'Mes {m}' for m in pivot.columns]
print(pivot.to_string())


Perfil de generación (mediana kWh) por mes y hora — solo 2025:
      Mes 1  Mes 2  Mes 3  Mes 4  Mes 5  Mes 6  Mes 7  Mes 8  Mes 9  Mes 10  Mes 11  Mes 12
Hora                                                                                       
0      0.00    0.0   0.00   0.00   0.00   0.00   0.00   0.00   0.00    0.00    0.00    0.00
1      0.00    0.0   0.00   0.00   0.00   0.00   0.00   0.00   0.00    0.00    0.00    0.00
2      0.00    0.0   0.00   0.00   0.00   0.00   0.00   0.00   0.00    0.00    0.00    0.00
3      0.00    0.0   0.00   0.00   0.00   0.00   0.00   0.00   0.00    0.00    0.00    0.00
4      0.00    0.0   0.00   0.00   0.00   0.00   0.00   0.00   0.00    0.00    0.00    0.00
5      0.00    0.0   0.00   0.00   0.00   0.00   0.00   0.00   0.00    0.00    0.00    0.00
6      0.00    0.0   0.00   0.00   0.00   0.00   0.00   0.00   0.00    0.00    0.05    0.00
7      0.49    0.0   0.02   0.68   1.34   0.68   0.91   0.66   0.34    0.23    2.59    1.16
8      3.79    0.

In [4]:
# Curva solar por mes — visualización del perfil
fig_perfil = go.Figure()
colores = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
           '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf','#aec7e8','#ffbb78']

for mes in range(1, 13):
    sub = perfil[perfil['Mes'] == mes]
    nombre_mes = ['','Ene','Feb','Mar','Abr','May','Jun',
                  'Jul','Ago','Sep','Oct','Nov','Dic'][mes]
    fig_perfil.add_trace(go.Scatter(
        x=sub['Hora'], y=sub['Gen_perfil'],
        mode='lines', name=nombre_mes,
        line=dict(color=colores[mes-1], width=1.5 if mes in [5,6,7] else 1.0,
                  dash='solid' if mes in [5,6,7] else 'dot')
    ))

fig_perfil.update_layout(
    title='<b>Perfil de Generación Solar por Mes — Mediana Histórica 2025</b>',
    xaxis_title='Hora del día', yaxis_title='Generación mediana (kWh)',
    plot_bgcolor='white', height=420,
    xaxis=dict(showgrid=True, gridcolor='#eee', dtick=1),
    yaxis=dict(showgrid=True, gridcolor='#eee'),
    legend=dict(orientation='h', y=-0.25, font=dict(size=10))
)
fig_perfil.show()


In [5]:
# Evaluar el perfil como modelo sobre el período de test (enero-mayo 2026)
df_test = df[df['Anio'] == 2026].merge(perfil, on=['Mes', 'Hora'], how='left')
df_test_sol = df_test[df_test['Gen_perfil'] > 0.5]  # solo horas con generacion esperada

rmse_p = np.sqrt(((df_test_sol['Generacion_kWh'] - df_test_sol['Gen_perfil'])**2).mean())
mae_p  = (df_test_sol['Generacion_kWh'] - df_test_sol['Gen_perfil']).abs().mean()
mask   = df_test_sol['Generacion_kWh'] > 1.0
mape_p = ((df_test_sol[mask]['Generacion_kWh'] - df_test_sol[mask]['Gen_perfil']).abs()
          / df_test_sol[mask]['Generacion_kWh']).mean() * 100
r2_p   = 1 - ((df_test_sol['Generacion_kWh'] - df_test_sol['Gen_perfil'])**2).sum() /              ((df_test_sol['Generacion_kWh'] - df_test_sol['Generacion_kWh'].mean())**2).sum()

print('Perfil histórico evaluado sobre 2026 (ene-may, horas con gen esperada > 0.5 kWh):')
print(f'  RMSE : {rmse_p:.3f} kWh')
print(f'  MAE  : {mae_p:.3f} kWh')
print(f'  MAPE : {mape_p:.1f}%')
print(f'  R²   : {r2_p:.4f}')
print()
print('Referencia: modelo de regresion con lags → RMSE=2.541 kWh, R²=0.94')


Perfil histórico evaluado sobre 2026 (ene-may, horas con gen esperada > 0.5 kWh):
  RMSE : 5.860 kWh
  MAE  : 4.076 kWh
  MAPE : 45.9%
  R²   : 0.7132

Referencia: modelo de regresion con lags → RMSE=2.541 kWh, R²=0.94


## 2b. Calibración Regional — Fanosa Veracruz (5 años)

In [6]:
import os

fanosa = pd.read_csv('FanosaVeracruz_generacioncompleto.csv', header=None,
                     names=['Planta', 'Fecha', 'Fanosa_kWh'])
fanosa['Fecha'] = pd.to_datetime(fanosa['Fecha'])
fanosa['Mes']   = fanosa['Fecha'].dt.month
fanosa['Anio']  = fanosa['Fecha'].dt.year

# Mediana mensual 5 años (excluimos 2020 parcial)
fanosa_5yr     = fanosa[fanosa['Anio'].isin([2021, 2022, 2023, 2024, 2025])]
fanosa_5yr_mes = fanosa_5yr.groupby('Mes')['Fanosa_kWh'].median()
fanosa_2025_ms = fanosa[fanosa['Anio'] == 2025].groupby('Mes')['Fanosa_kWh'].median()

# Anomalia de 2025 vs historico regional (misma temporada de lluvias Veracruz)
anomalia = (fanosa_2025_ms / fanosa_5yr_mes).dropna()
factor   = (1.0 / anomalia).clip(0.5, 2.0)  # max +/-50% correccion
fdict    = factor.to_dict()

nombres_m = ['','Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
print(f'{"Mes":<6} {"Fanosa 5yr":>12} {"Fanosa 2025":>12} {"Anomalia":>10} {"Factor":>8}')
print('-' * 52)
for mes in sorted(fanosa_5yr_mes.index):
    nom  = nombres_m[mes]
    f5   = fanosa_5yr_mes.get(mes, 0)
    f25  = fanosa_2025_ms.get(mes, float('nan'))
    anom = anomalia.get(mes, float('nan')) if mes in anomalia.index else float('nan')
    fc   = fdict.get(mes, 1.0)
    print(f'{nom:<6} {f5:>12.1f} {f25:>12.1f} {anom:>10.3f} {fc:>8.3f}')

# Aplicar factor al perfil de El Viejon
perfil_corr = perfil.copy()
perfil_corr['Gen_perfil'] = perfil_corr.apply(
    lambda r: r['Gen_perfil'] * fdict.get(int(r['Mes']), 1.0), axis=1
)

# Comparar los dos perfiles sobre datos 2026
df_t26 = df[df['Anio'] == 2026]
best_rmse = 1e9
perfil_final = perfil
perfil_final_label = 'Original 2025'
print()
print('Comparacion de perfiles en validacion 2026:')
for label, perf in [('Original 2025', perfil), ('Calibrado Fanosa', perfil_corr)]:
    dv = df_t26.merge(perf.rename(columns={'Gen_perfil': 'GP'}),
                      on=['Mes', 'Hora'], how='left')
    dv = dv[dv['GP'] > 0.5]
    rmse = np.sqrt(((dv['Generacion_kWh'] - dv['GP']) ** 2).mean())
    r2   = 1 - ((dv['Generacion_kWh'] - dv['GP']) ** 2).sum() / \
               ((dv['Generacion_kWh'] - dv['Generacion_kWh'].mean()) ** 2).sum()
    mask = dv['Generacion_kWh'] > 1.0
    mape = ((dv[mask]['Generacion_kWh'] - dv[mask]['GP']).abs() /
            dv[mask]['Generacion_kWh']).mean() * 100
    print(f'  {label:20s}  RMSE={rmse:.3f}  MAPE={mape:.1f}%  R2={r2:.4f}')
    if rmse < best_rmse:
        best_rmse  = rmse
        perfil_final = perf.copy()
        perfil_final_label = label

print(f'\n=> Usando: {perfil_final_label}')


Mes      Fanosa 5yr  Fanosa 2025   Anomalia   Factor
----------------------------------------------------
Ene           594.9        603.2      1.014    0.986
Feb           721.0        626.2      0.868    1.151
Mar           845.9        791.5      0.936    1.069
Abr           914.1       1029.6      1.126    0.888
May           827.3        990.9      1.198    0.835
Jun           968.8        875.6      0.904    1.107
Jul          1112.4        961.6      0.864    1.157
Ago          1099.6       1034.2      0.941    1.063
Sep           917.3        936.9      1.021    0.979
Oct           838.4        921.8      1.100    0.910
Nov           721.7          nan        nan    1.000
Dic           655.3          nan        nan    1.000

Comparacion de perfiles en validacion 2026:
  Original 2025         RMSE=5.860  MAPE=45.9%  R2=0.7132
  Calibrado Fanosa      RMSE=6.185  MAPE=45.3%  R2=0.6728

=> Usando: Original 2025


## 3. Análogo Histórico — Opción 1

In [7]:
FECHA_CORTE = df['Datetime'].max()
hace_2s   = FECHA_CORTE - pd.Timedelta(weeks=2)
hace_1a   = FECHA_CORTE - pd.DateOffset(years=1)
hace_1a2s = hace_1a    - pd.Timedelta(weeks=2)

gen_rec  = df[(df['Datetime'] >= hace_2s)]['Generacion_kWh'].mean()
gen_anio = df[(df['Datetime'] >= hace_1a2s) & (df['Datetime'] <= hace_1a)]['Generacion_kWh'].mean()
ratio    = gen_rec / gen_anio if gen_anio > 0 else 1.0

print(f'Último dato        : {FECHA_CORTE.date()}')
print(f'Gen últimas 2 sem  : {gen_rec:.3f} kWh/h')
print(f'Gen mismas 2 sem -1a: {gen_anio:.3f} kWh/h')
print(f'Ratio rendimiento  : {ratio:.3f}  ({(ratio-1)*100:+.1f}% vs año anterior)')


Último dato        : 2026-05-17
Gen últimas 2 sem  : 9.221 kWh/h
Gen mismas 2 sem -1a: 8.325 kWh/h
Ratio rendimiento  : 1.108  (+10.8% vs año anterior)


In [8]:
fecha_ini_fc = pd.Timestamp('2026-05-18 00:00:00')
fecha_fin_fc = pd.Timestamp('2026-07-31 23:00:00')
futuras = pd.date_range(fecha_ini_fc, fecha_fin_fc, freq='h')

rows_ana = []
for dt in futuras:
    for delta in [1, 2]:
        dt_ref = dt - pd.DateOffset(years=delta)
        mask = df['Datetime'] == dt_ref
        if mask.any():
            rows_ana.append({
                'Datetime': dt, 'Mes': dt.month, 'Hora': dt.hour,
                'Gen_analogo': max(df.loc[mask, 'Generacion_kWh'].values[0] * ratio, 0),
                'Fuente': f'-{delta}a'
            })
            break
    else:
        rows_ana.append({'Datetime': dt, 'Mes': dt.month, 'Hora': dt.hour,
                         'Gen_analogo': 0.0, 'Fuente': 'sin_dato'})

df_ana = pd.DataFrame(rows_ana)
print(df_ana['Fuente'].value_counts().to_string())
print()
for mes in [5, 6, 7]:
    sub = df_ana[df_ana['Mes'] == mes]
    dias = sub['Datetime'].dt.date.nunique()
    tot = sub['Gen_analogo'].sum()
    print(f'Mes {mes}: {tot:.0f} kWh | {tot/dias:.1f} kWh/día')


Fuente
-1a    1800

Mes 5: 3133 kWh | 223.8 kWh/día
Mes 6: 5860 kWh | 195.3 kWh/día
Mes 7: 7563 kWh | 244.0 kWh/día


## 4. Forecast Combinado (60% Perfil + 40% Análogo)

In [9]:
# Perfil histórico escalado por ratio de rendimiento
df_fc = pd.DataFrame({'Datetime': futuras})
df_fc['Mes']  = df_fc['Datetime'].dt.month
df_fc['Hora'] = df_fc['Datetime'].dt.hour
df_fc = df_fc.merge(perfil_final, on=['Mes', 'Hora'], how='left')
df_fc['Gen_perfil'] = (df_fc['Gen_perfil'] * ratio).fillna(0.0)

# Unir análogo
df_fc = df_fc.merge(df_ana[['Datetime','Gen_analogo']], on='Datetime', how='left')
df_fc['Gen_analogo'] = df_fc['Gen_analogo'].fillna(0.0)

# Combinado
df_fc['Gen_combinada'] = 0.6 * df_fc['Gen_perfil'] + 0.4 * df_fc['Gen_analogo']

nombres_mes = {5:'Mayo', 6:'Junio', 7:'Julio'}
print(f'{"Mes":<10} {"Perfil kWh":>11} {"Análogo kWh":>12} {"Combinado kWh":>14} {"kWh/día":>8}')
print('-' * 58)
for mes in [5, 6, 7]:
    sub  = df_fc[df_fc['Mes'] == mes]
    dias = sub['Datetime'].dt.date.nunique()
    t_p  = sub['Gen_perfil'].sum()
    t_a  = sub['Gen_analogo'].sum()
    t_c  = sub['Gen_combinada'].sum()
    print(f'{nombres_mes[mes]:<10} {t_p:>11.0f} {t_a:>12.0f} {t_c:>14.0f} {t_c/dias:>8.1f}')


Mes         Perfil kWh  Análogo kWh  Combinado kWh  kWh/día
----------------------------------------------------------
Mayo              3406         3133           3297    235.5
Junio             6278         5860           6111    203.7
Julio             8442         7563           8091    261.0


## 5. Gráficas

In [10]:
hist_rec = df[df['Datetime'] >= '2026-04-18'][['Datetime','Generacion_kWh']]

fig_serie = go.Figure()
fig_serie.add_trace(go.Scatter(
    x=hist_rec['Datetime'], y=hist_rec['Generacion_kWh'],
    mode='lines', name='Histórico real', line=dict(color='#2c2c2c', width=1.2),
    hovertemplate='%{x|%d %b %H:%M}<br>Real: %{y:.1f} kWh<extra></extra>'
))
fig_serie.add_trace(go.Scatter(
    x=df_fc['Datetime'], y=df_fc['Gen_perfil'],
    mode='lines', name='Perfil histórico', line=dict(color='#1f77b4', width=1.0, dash='dot'),
    hovertemplate='%{x|%d %b %H:%M}<br>Perfil: %{y:.1f} kWh<extra></extra>'
))
fig_serie.add_trace(go.Scatter(
    x=df_fc['Datetime'], y=df_fc['Gen_analogo'],
    mode='lines', name='Análogo -1 año', line=dict(color='#2ca02c', width=1.0, dash='dash'),
    hovertemplate='%{x|%d %b %H:%M}<br>Análogo: %{y:.1f} kWh<extra></extra>'
))
fig_serie.add_trace(go.Scatter(
    x=df_fc['Datetime'], y=df_fc['Gen_combinada'],
    mode='lines', name='Combinado (60% perfil + 40% análogo)', line=dict(color='#d62728', width=2),
    hovertemplate='%{x|%d %b %H:%M}<br>Combinado: %{y:.1f} kWh<extra></extra>'
))
fig_serie.add_vline(x='2026-05-18', line_dash='dash', line_color='gray', line_width=1)
fig_serie.update_layout(
    title='<b>Forecast Solar — FERCHEGAS El Viejón: Mayo 18 – Julio 31, 2026</b>',
    xaxis_title='Fecha', yaxis_title='Generación (kWh)',
    plot_bgcolor='white', height=460,
    xaxis=dict(showgrid=True, gridcolor='#eee'),
    yaxis=dict(showgrid=True, gridcolor='#eee'),
    legend=dict(orientation='h', y=-0.2)
)
fig_serie.show()


In [11]:
# Generación diaria
daily_fc = df_fc.groupby(df_fc['Datetime'].dt.date).agg(
    Gen_perfil=('Gen_perfil','sum'),
    Gen_analogo=('Gen_analogo','sum'),
    Gen_combinada=('Gen_combinada','sum')
).reset_index()
daily_fc.columns = ['Fecha','Gen_perfil','Gen_analogo','Gen_combinada']
daily_fc['Fecha'] = pd.to_datetime(daily_fc['Fecha'])
daily_fc['Mes']   = daily_fc['Fecha'].dt.month

colores_mes = {5:'#ff7f0e', 6:'#2ca02c', 7:'#9467bd'}
abr = df[df['Mes'] == 4]
media_abr = abr['Generacion_kWh'].sum() / abr['Dia'].nunique()

fig_barras = go.Figure()
for mes in [5, 6, 7]:
    sub = daily_fc[daily_fc['Mes'] == mes]
    fig_barras.add_trace(go.Bar(
        x=sub['Fecha'], y=sub['Gen_combinada'], name=nombres_mes[mes],
        marker_color=colores_mes[mes], opacity=0.85,
        hovertemplate='%{x|%d %b}<br>%{y:.1f} kWh<extra></extra>'
    ))
fig_barras.add_hline(y=media_abr, line_dash='dash', line_color='red', line_width=1.5)
fig_barras.add_annotation(
    x=daily_fc['Fecha'].max(), y=media_abr,
    text=f'Media diaria abril real: {media_abr:.0f} kWh',
    showarrow=False, xanchor='right', yanchor='bottom', font=dict(color='red', size=11)
)
fig_barras.update_layout(
    title='<b>Generación Diaria Estimada — Forecast Combinado</b>',
    xaxis_title='Fecha', yaxis_title='kWh/día',
    plot_bgcolor='white', height=430,
    xaxis=dict(showgrid=True, gridcolor='#eee'),
    yaxis=dict(showgrid=True, gridcolor='#eee'),
    legend=dict(orientation='h', y=-0.2)
)
fig_barras.show()


In [12]:
# Curva solar promedio por mes
df_fc['Hora'] = df_fc['Datetime'].dt.hour
curva_fc = df_fc.groupby(['Mes','Hora'])['Gen_combinada'].mean().reset_index()

fig_curva = go.Figure()
for anio_ref, color, dash in [(2024,'#aec7e8','dot'),(2025,'#2c2c2c','dash')]:
    for mes_ref in [5, 6, 7]:
        curva_r = df[(df['Mes']==mes_ref) & (df['Anio']==anio_ref)].groupby('Hora')['Generacion_kWh'].mean().reset_index()
        if len(curva_r):
            fig_curva.add_trace(go.Scatter(
                x=curva_r['Hora'], y=curva_r['Generacion_kWh'],
                mode='lines', name=f'{nombres_mes[mes_ref]} {anio_ref}',
                line=dict(color=colores_mes[mes_ref], width=1.0, dash=dash), opacity=0.6
            ))

for mes in [5, 6, 7]:
    sub = curva_fc[curva_fc['Mes']==mes]
    fig_curva.add_trace(go.Scatter(
        x=sub['Hora'], y=sub['Gen_combinada'],
        mode='lines+markers', name=f'{nombres_mes[mes]} 2026 (forecast)',
        line=dict(color=colores_mes[mes], width=2.5), marker=dict(size=5)
    ))
fig_curva.update_layout(
    title='<b>Curva Solar Promedio — Forecast 2026 vs Histórico Real</b>',
    xaxis_title='Hora del día', yaxis_title='Generación promedio (kWh)',
    plot_bgcolor='white', height=440,
    xaxis=dict(showgrid=True, gridcolor='#eee', dtick=1),
    yaxis=dict(showgrid=True, gridcolor='#eee'),
    legend=dict(orientation='h', y=-0.25, font=dict(size=10))
)
fig_curva.show()


## 6. Resumen y Métricas

In [13]:
# Precisión del perfil histórico sobre datos conocidos (2026 ene-may)
df_val = df[df['Anio'] == 2026].merge(perfil, on=['Mes','Hora'], how='left')
df_val_sol = df_val[df_val['Gen_perfil'] > 0.5]
rmse_v = np.sqrt(((df_val_sol['Generacion_kWh'] - df_val_sol['Gen_perfil'])**2).mean())
r2_v   = 1 - ((df_val_sol['Generacion_kWh'] - df_val_sol['Gen_perfil'])**2).sum() /              ((df_val_sol['Generacion_kWh'] - df_val_sol['Generacion_kWh'].mean())**2).sum()
mask_v = df_val_sol['Generacion_kWh'] > 1.0
mape_v = ((df_val_sol[mask_v]['Generacion_kWh'] - df_val_sol[mask_v]['Gen_perfil']).abs()
          / df_val_sol[mask_v]['Generacion_kWh']).mean() * 100

print('=' * 65)
print('  FORECAST SOLAR — FERCHEGAS EL VIEJÓN')
print('  Perfil Histórico (60%) + Análogo -1 año escalado (40%)')
print('=' * 65)
print()
print(f'  Ratio rendimiento reciente : {ratio:.3f}  ({(ratio-1)*100:+.1f}%)')
print(f'  Período histórico del perfil: 2025 (primer año completo de operación)')
print()
print('  Precisión del perfil en ene–may 2026 (validación):')
print(f'    RMSE : {rmse_v:.3f} kWh   (vs modelo lag RMSE=2.541 kWh)')
print(f'    MAPE : {mape_v:.1f}%')
print(f'    R²   : {r2_v:.4f}')
print()
print(f'  {"Período":<22} {"kWh total":>10} {"kWh/día":>9} {"vs Abr 2026":>12}')
print('  ' + '-' * 55)

abr_tot  = df[df['Mes']==4]['Generacion_kWh'].sum()
abr_dias = df[df['Mes']==4]['Dia'].nunique()
print(f'  {"Abril 2026 (REAL)":<22} {abr_tot:>10.0f} {abr_tot/abr_dias:>9.1f} {"—":>12}')

for mes, nombre in [(5,'Mayo 18–31'),(6,'Junio 2026'),(7,'Julio 2026')]:
    sub  = df_fc[df_fc['Mes']==mes]
    dias = sub['Datetime'].dt.date.nunique()
    tot  = sub['Gen_combinada'].sum()
    vs_abr = (tot/dias - abr_tot/abr_dias) / (abr_tot/abr_dias) * 100
    print(f'  {nombre:<22} {tot:>10.0f} {tot/dias:>9.1f} {vs_abr:>+11.1f}%')

print()
print('  Nota: junio-julio son temporada de lluvias en Veracruz.')
print('  El perfil histórico ya incorpora la reduccion real por nubosidad.')


  FORECAST SOLAR — FERCHEGAS EL VIEJÓN
  Perfil Histórico (60%) + Análogo -1 año escalado (40%)

  Ratio rendimiento reciente : 1.108  (+10.8%)
  Período histórico del perfil: 2025 (primer año completo de operación)

  Precisión del perfil en ene–may 2026 (validación):
    RMSE : 5.860 kWh   (vs modelo lag RMSE=2.541 kWh)
    MAPE : 45.9%
    R²   : 0.7132

  Período                 kWh total   kWh/día  vs Abr 2026
  -------------------------------------------------------
  Abril 2026 (REAL)           12776     142.0            —
  Mayo 18–31                   3297     235.5       +65.9%
  Junio 2026                   6111     203.7       +43.5%
  Julio 2026                   8091     261.0       +83.9%

  Nota: junio-julio son temporada de lluvias en Veracruz.
  El perfil histórico ya incorpora la reduccion real por nubosidad.


## GBM Diario + Clima + Tendencia

Mismo enfoque del dashboard: GBM entrenado en 2025, corregido con ratio de rendimiento actual, irradiación diaria real del CSV de clima y tendencia lineal 2026.

In [14]:
from sklearn.ensemble import GradientBoostingRegressor
from scipy import stats as _stats

# ── Irradiación diaria ────────────────────────────────────────────────────────
_clim = pd.read_csv('Clima_ferchegaselviejon.csv', header=None,
    names=['id','np','id2','irr','hum','vv','nub','temp','fec','hor','fh','gen'])
_clim['fh']  = pd.to_datetime(_clim['fh'], errors='coerce')
_clim['irr'] = pd.to_numeric(_clim['irr'], errors='coerce').fillna(0)
_clim = _clim.dropna(subset=['fh']).drop_duplicates('fh').set_index('fh').sort_index()
irr_d = _clim['irr'].resample('D').sum().rename('irr_dia').reset_index()
irr_d.columns = ['Fecha', 'irr_dia']

# ── Datos diarios para GBM ────────────────────────────────────────────────────
daily_m = df.copy()
daily_m['Fecha'] = pd.to_datetime(daily_m['Datetime'].dt.date)
daily_m = daily_m.groupby('Fecha').agg(Gen_kWh=('Generacion_kWh','sum')).reset_index()
daily_m['Anio'] = daily_m['Fecha'].dt.year
daily_m['Mes']  = daily_m['Fecha'].dt.month
daily_m['dow']  = daily_m['Fecha'].dt.dayofweek
daily_m = daily_m.merge(irr_d, on='Fecha', how='left')
daily_m['irr_dia'] = daily_m['irr_dia'].fillna(0)

# Lags y rolling
for lag in [1, 2, 3, 7]:
    daily_m[f'lag{lag}d'] = daily_m['Gen_kWh'].shift(lag)
daily_m['irr_lag1']    = daily_m['irr_dia'].shift(1)
daily_m['roll_mean7']  = daily_m['Gen_kWh'].shift(1).rolling(7).mean()
daily_m['roll_std7']   = daily_m['Gen_kWh'].shift(1).rolling(7).std()
daily_m['roll_mean14'] = daily_m['Gen_kWh'].shift(1).rolling(14).mean()
daily_m['Target']      = daily_m['Gen_kWh'].shift(-1)

FEATS_D = ['lag1d','lag2d','lag3d','lag7d',
           'roll_mean7','roll_std7','roll_mean14',
           'irr_dia','irr_lag1','Mes','dow']
dm_d = daily_m.dropna(subset=FEATS_D + ['Target']).reset_index(drop=True)

# ── Entrenar con 2025 ─────────────────────────────────────────────────────────
tr_d = dm_d[dm_d['Anio'] == 2025]
gbm_d = GradientBoostingRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                                   subsample=0.8, min_samples_leaf=5, random_state=42)
gbm_d.fit(tr_d[FEATS_D], tr_d['Target'])

# ── Ratio de corrección: GBM vs real en 2026 conocido ─────────────────────────
known_2026 = dm_d[(dm_d['Anio'] == 2026) &
                  (dm_d['Fecha'] <= FECHA_CORTE.normalize())].copy()
if len(known_2026) > 0:
    known_2026['pred'] = gbm_d.predict(known_2026[FEATS_D]).clip(min=0)
    ratio_gbm = known_2026['Gen_kWh'].sum() / max(known_2026['pred'].sum(), 1)
else:
    ratio_gbm = 1.0

# ── Tendencia lineal sobre datos 2026 ─────────────────────────────────────────
d26 = daily_m[daily_m['Anio'] == 2026].dropna(subset=['Gen_kWh']).copy()
d26['t'] = (d26['Fecha'] - d26['Fecha'].min()).dt.days
if len(d26) >= 5:
    slope26, _, _, p26, _ = _stats.linregress(d26['t'], d26['Gen_kWh'])
    fecha_ref_tend = d26['Fecha'].max()
    dias_ref_tend  = int((fecha_ref_tend - d26['Fecha'].min()).days)
else:
    slope26, p26 = 0.0, 1.0
    fecha_ref_tend = pd.Timestamp('2026-05-17')
    dias_ref_tend  = 0

# ── Irradiación de referencia por mes (promedio mismo mes año anterior) ────────
irr_lookup = irr_d.set_index('Fecha')['irr_dia'].to_dict()
irr_ref_mes = {}
for mes in [5, 6, 7]:
    prev = irr_d[(irr_d['Fecha'].dt.year == 2025) & (irr_d['Fecha'].dt.month == mes)]
    irr_ref_mes[mes] = float(prev['irr_dia'].mean()) if len(prev) > 0 else 1.0

print(f'Ratio corrección GBM : {ratio_gbm:.3f}x')
print(f'Tendencia 2026       : {slope26:+.2f} kWh/día/día  (p={p26:.4f})')
for mes in [5, 6, 7]:
    print(f'Irr. ref {nombres_mes[mes]} 2025  : {irr_ref_mes[mes]:.0f} unid/día')


Ratio corrección GBM : 1.151x
Tendencia 2026       : +1.11 kWh/día/día  (p=0.0000)
Irr. ref Mayo 2025  : 6256 unid/día
Irr. ref Junio 2025  : 5184 unid/día
Irr. ref Julio 2025  : 6631 unid/día


In [15]:
# ── Forecast iterativo día a día: Mayo 18 – Julio 31 ─────────────────────────
hist_gen = list(daily_m[daily_m['Fecha'] <= FECHA_CORTE.normalize()]
                        .tail(14)['Gen_kWh'].values)

preds_gbm  = []
fc_gbm_lst = []
current = pd.Timestamp('2026-05-18')
fecha_fin_d = pd.Timestamp('2026-07-31')

while current <= fecha_fin_d:
    all_gen = hist_gen + preds_gbm

    lag1 = all_gen[-1]  if len(all_gen) >= 1  else 0
    lag2 = all_gen[-2]  if len(all_gen) >= 2  else 0
    lag3 = all_gen[-3]  if len(all_gen) >= 3  else 0
    lag7 = all_gen[-7]  if len(all_gen) >= 7  else 0
    rm7  = float(np.mean(all_gen[-7:]))  if len(all_gen) >= 7  else float(np.mean(all_gen))
    rs7  = float(np.std(all_gen[-7:]))   if len(all_gen) >= 7  else 0.0
    rm14 = float(np.mean(all_gen[-14:])) if len(all_gen) >= 14 else float(np.mean(all_gen))

    # Irradiación: usar dato real del CSV si existe, si no, promedio del mes anterior
    irr_hoy  = irr_lookup.get(current,
               irr_ref_mes.get(current.month, 1.0))
    irr_ayer = irr_lookup.get(current - pd.Timedelta(days=1),
               irr_ref_mes.get(current.month, 1.0))

    x = np.array([[lag1, lag2, lag3, lag7,
                   rm7, rs7, rm14,
                   irr_hoy, irr_ayer,
                   current.month, current.dayofweek]])

    pred = max(float(gbm_d.predict(x)[0]) * ratio_gbm, 0.0)

    # Corrección de tendencia
    if len(d26) >= 5:
        dias_fut   = int((current - d26['Fecha'].min()).days)
        delta_tend = slope26 * (dias_fut - dias_ref_tend)
        pred = max(pred + delta_tend, 0.0)

    preds_gbm.append(pred)
    fc_gbm_lst.append({'Fecha': current, 'Gen_GBM': pred, 'Mes': current.month})
    current += pd.Timedelta(days=1)

df_gbm_fc = pd.DataFrame(fc_gbm_lst)
df_gbm_fc['Fecha'] = pd.to_datetime(df_gbm_fc['Fecha'])

# Agregar al daily_fc existente
if 'Gen_GBM' in daily_fc.columns:
    daily_fc = daily_fc.drop(columns=['Gen_GBM'])
daily_fc = daily_fc.merge(df_gbm_fc[['Fecha','Gen_GBM']], on='Fecha', how='left')

print(f'{"Mes":<8} {"Combinado kWh/día":>18} {"GBM+Clima kWh/día":>18} {"Dif%":>6}')
print('-' * 55)
for mes in [5, 6, 7]:
    sc = daily_fc[daily_fc['Mes'] == mes]['Gen_combinada'].mean()
    sg = daily_fc[daily_fc['Mes'] == mes]['Gen_GBM'].mean()
    dif = (sg/sc - 1)*100 if sc > 0 else 0
    print(f'{nombres_mes[mes]:<8} {sc:>18.1f} {sg:>18.1f} {dif:>+6.1f}%')


Mes       Combinado kWh/día  GBM+Clima kWh/día   Dif%
-------------------------------------------------------
Mayo                  235.5              228.2   -3.1%
Junio                 203.7              246.1  +20.8%
Julio                 261.0              271.5   +4.0%


In [16]:
# ── Gráfico: barras Combinado + línea GBM por mes ─────────────────────────────
colores_base = {5:'#ff7f0e', 6:'#2ca02c', 7:'#9467bd'}
colores_gbm  = {5:'#cc5500', 6:'#1a7a1a', 7:'#6a3d9a'}

fig_gbm = go.Figure()
for mes in [5, 6, 7]:
    sub = daily_fc[daily_fc['Mes'] == mes]
    # Barras del modelo combinado (fondo)
    fig_gbm.add_trace(go.Bar(
        x=sub['Fecha'], y=sub['Gen_combinada'],
        name=f'{nombres_mes[mes]} — Combinado',
        marker_color=colores_base[mes], opacity=0.45,
        hovertemplate='%{x|%d %b}<br>Combinado: %{y:.0f} kWh<extra></extra>'))
    # Línea GBM encima
    fig_gbm.add_trace(go.Scatter(
        x=sub['Fecha'], y=sub['Gen_GBM'],
        name=f'{nombres_mes[mes]} — GBM+Clima',
        line=dict(color=colores_gbm[mes], width=2.2),
        mode='lines+markers', marker=dict(size=4),
        hovertemplate='%{x|%d %b}<br>GBM: %{y:.0f} kWh<extra></extra>'))

fig_gbm.add_hline(y=media_abr, line_dash='dash', line_color='red', line_width=1.5,
    annotation_text=f'Media diaria abril real: {media_abr:.0f} kWh',
    annotation_position='top left', annotation_font=dict(color='red', size=11))

fig_gbm.update_layout(
    title=(f'<b>GBM Diario + Clima + Tendencia — Forecast Mayo–Julio 2026</b><br>'
           f'<span style="font-size:11px;color:#555">'
           f'Ratio corrección: {ratio_gbm:.3f}x  ·  '
           f'Tendencia: {slope26:+.2f} kWh/día/día  ·  '
           f'Barras = Combinado (perfil+análogo) | Línea = GBM+Clima</span>'),
    xaxis=dict(title='Fecha', showgrid=True, gridcolor='#eee',
               tickformat='%d %b', dtick='D2', tickangle=-35),
    yaxis=dict(title='kWh/día', rangemode='tozero',
               showgrid=True, gridcolor='#eee'),
    plot_bgcolor='white', height=470, barmode='overlay',
    legend=dict(orientation='h', y=-0.28, font=dict(size=10)))
fig_gbm.show()


In [17]:
# ── Importancia de variables del modelo diario ────────────────────────────────
imp = pd.Series(gbm_d.feature_importances_, index=FEATS_D).sort_values(ascending=True)
etiquetas = {
    'lag1d':'Gen ayer', 'lag2d':'Gen hace 2d', 'lag3d':'Gen hace 3d',
    'lag7d':'Gen hace 7d', 'roll_mean7':'Media 7d', 'roll_std7':'Std 7d',
    'roll_mean14':'Media 14d', 'irr_dia':'Irradiación hoy',
    'irr_lag1':'Irradiación ayer', 'Mes':'Mes', 'dow':'Día semana'}
imp.index = [etiquetas.get(f, f) for f in imp.index]

fig_imp = go.Figure(go.Bar(
    x=imp.values, y=imp.index, orientation='h',
    marker_color=['#1B5E20' if i >= len(imp)-3 else '#66BB6A'
                  for i in range(len(imp))],
    text=[f'{v:.3f}' for v in imp.values], textposition='outside'))
fig_imp.update_layout(
    title='<b>Importancia de variables — Modelo GBM Diario</b>',
    xaxis=dict(title='Importancia (Gini)', showgrid=True, gridcolor='#eee'),
    yaxis=dict(tickfont=dict(size=11)),
    plot_bgcolor='white', height=420,
    margin=dict(l=130, r=60, t=50, b=30))
fig_imp.show()


## Validación — Real vs Pronóstico Oficial vs GBM (18–24 mayo 2026)

Comparación con los datos reales del período ya transcurrido.

In [18]:
import glob as _glob

# Cargar archivo de generacion real May 18-24
_files = _glob.glob(r'c:\Users\ajasa\OneDrive\Documentos\Graficos\Reporte_generacion_FERCHEGAS*18*may*.xls')
if not _files:
    raise FileNotFoundError('No se encontro el archivo Reporte_generacion May 18-24')
_rpt = pd.read_html(_files[0])[0]
_rpt.columns = ['Fecha', 'Real_kWh', 'Pron_kWh']
_rpt['Fecha'] = pd.to_datetime(_rpt['Fecha'], errors='coerce')
_rpt['Real_kWh'] = pd.to_numeric(_rpt['Real_kWh'], errors='coerce')
_rpt['Pron_kWh'] = pd.to_numeric(_rpt['Pron_kWh'], errors='coerce')
_rpt = _rpt.dropna(subset=['Fecha']).reset_index(drop=True)

# Unir con forecast GBM del notebook
_val = _rpt.merge(
    daily_fc[['Fecha', 'Gen_combinada', 'Gen_GBM']],
    on='Fecha', how='left')

# Errores
_val['Err_Pron']  = _val['Pron_kWh']    - _val['Real_kWh']   # positivo = sobreestima
_val['Err_GBM']   = _val['Gen_GBM']      - _val['Real_kWh']
_val['Err_Comb']  = _val['Gen_combinada'] - _val['Real_kWh']
_val['Mejor'] = _val.apply(
    lambda r: 'GBM' if abs(r['Err_GBM']) < abs(r['Err_Pron']) else 'Pronóstico',
    axis=1)

# Fila totales / promedios
_totales = {
    'Fecha': pd.Timestamp('2026-01-01'),   # placeholder, no se muestra
    'Real_kWh':      _val['Real_kWh'].sum(),
    'Pron_kWh':      _val['Pron_kWh'].sum(),
    'Gen_GBM':       _val['Gen_GBM'].sum(),
    'Gen_combinada': _val['Gen_combinada'].sum(),
    'Err_Pron':      _val['Err_Pron'].sum(),
    'Err_GBM':       _val['Err_GBM'].sum(),
    'Err_Comb':      _val['Err_Comb'].sum(),
    'Mejor': 'GBM' if abs(_val['Err_GBM'].sum()) < abs(_val['Err_Pron'].sum()) else 'Pronóstico',
}

# ── Tabla Plotly ─────────────────────────────────────────────────────────
VERDE     = '#1B5E20'
VERDE_MED = '#2E7D32'
VERDE_L   = '#A5D6A7'
ROJO      = '#FFCDD2'
NARANJA   = '#FFE0B2'
GRIS      = '#ECEFF1'

nombres_mes_v = ['','Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

def fmt_fecha(ts):
    return ts.strftime('%a %d %b').capitalize()

def color_err(val):
    if abs(val) < 10:   return 'rgba(165,214,167,0.6)'
    if abs(val) < 30:   return 'rgba(255,224,178,0.7)'
    return 'rgba(255,205,210,0.8)'

dias_lbl   = [fmt_fecha(r['Fecha']) for _, r in _val.iterrows()] + ['TOTAL 7 días']
real_vals  = [f"{r['Real_kWh']:.1f}" for _, r in _val.iterrows()] + [f"{_totales['Real_kWh']:.1f}"]
pron_vals  = [f"{r['Pron_kWh']:.1f}" for _, r in _val.iterrows()] + [f"{_totales['Pron_kWh']:.1f}"]
gbm_vals   = [f"{r['Gen_GBM']:.1f}" for _, r in _val.iterrows()] + [f"{_totales['Gen_GBM']:.1f}"]
comb_vals  = [f"{r['Gen_combinada']:.1f}" for _, r in _val.iterrows()] + [f"{_totales['Gen_combinada']:.1f}"]
ep_vals    = [f"{r['Err_Pron']:+.1f}" for _, r in _val.iterrows()] + [f"{_totales['Err_Pron']:+.1f}"]
eg_vals    = [f"{r['Err_GBM']:+.1f}" for _, r in _val.iterrows()] + [f"{_totales['Err_GBM']:+.1f}"]
ec_vals    = [f"{r['Err_Comb']:+.1f}" for _, r in _val.iterrows()] + [f"{_totales['Err_Comb']:+.1f}"]
mejor_vals = list(_val['Mejor']) + [_totales['Mejor']]

# Colores por celda
n = len(dias_lbl)
col_fecha  = ['rgba(46,125,50,0.12)']*n
col_fecha[-1] = 'rgba(46,125,50,0.25)'
col_real   = ['white']*n; col_real[-1] = 'rgba(46,125,50,0.15)'
col_pron   = ['white']*n; col_pron[-1] = 'rgba(46,125,50,0.15)'
col_gbm    = ['white']*n; col_gbm[-1]  = 'rgba(46,125,50,0.15)'
col_comb   = ['white']*n; col_comb[-1] = 'rgba(46,125,50,0.15)'

col_ep = [color_err(r['Err_Pron']) for _, r in _val.iterrows()] + [color_err(_totales['Err_Pron'])]
col_eg = [color_err(r['Err_GBM'])  for _, r in _val.iterrows()] + [color_err(_totales['Err_GBM'])]
col_ec = [color_err(r['Err_Comb']) for _, r in _val.iterrows()] + [color_err(_totales['Err_Comb'])]

col_mejor = [
    'rgba(165,214,167,0.7)' if m == 'GBM' else 'rgba(255,224,178,0.7)'
    for m in mejor_vals
]

import plotly.graph_objects as go
fig_val = go.Figure(go.Table(
    columnwidth=[90, 65, 65, 70, 75, 75, 75, 80],
    header=dict(
        values=['<b>Fecha</b>',
                '<b>Real<br>(kWh)</b>',
                '<b>Pronóstico<br>oficial (kWh)</b>',
                '<b>GBM+Clima<br>(kWh)</b>',
                '<b>Δ Pronóstico<br>(kWh)</b>',
                '<b>Δ GBM<br>(kWh)</b>',
                '<b>Mejor</b>'],
        fill_color=VERDE,
        font=dict(color='white', size=11),
        align=['left','center','center','center','center','center','center'],
        height=36),
    cells=dict(
        values=[dias_lbl, real_vals, pron_vals, gbm_vals,
                ep_vals, eg_vals, mejor_vals],
        fill_color=[col_fecha, col_real, col_pron, col_gbm,
                    col_ep, col_eg, col_mejor],
        font=dict(color='#111', size=11),
        align=['left','center','center','center','center','center','center'],
        height=30)))

# MAPE de cada modelo
_mape_pron = (_val['Err_Pron'].abs() / _val['Real_kWh'] * 100).mean()
_mape_gbm  = (_val['Err_GBM'].abs()  / _val['Real_kWh'] * 100).mean()
_mape_comb = (_val['Err_Comb'].abs() / _val['Real_kWh'] * 100).mean()

fig_val.update_layout(
    title=(
        '<b>Validación: Real vs Pronóstico Oficial vs GBM — 18 al 24 mayo 2026</b><br>'
        f'<span style="font-size:10px;color:#555">'
        f'Δ = Predicho − Real (positivo = sobreestimado, negativo = subestimado) | '
        f'MAPE Pronóstico oficial: {_mape_pron:.1f}% | '
        f'MAPE GBM+Clima: {_mape_gbm:.1f}%</span>'),
    height=380,
    margin=dict(l=0, r=0, t=80, b=0))
fig_val.show()

# ── Gráfico de barras comparativo ─────────────────────────────────────────
fig_bars = go.Figure()
fechas_v  = _val['Fecha']
fig_bars.add_trace(go.Bar(
    x=fechas_v, y=_val['Real_kWh'],
    name='Real', marker_color='#1B5E20', opacity=0.9,
    hovertemplate='%{x|%d %b}<br>Real: %{y:.1f} kWh<extra></extra>'))
fig_bars.add_trace(go.Bar(
    x=fechas_v, y=_val['Pron_kWh'],
    name='Pronóstico oficial', marker_color='#1565C0', opacity=0.7,
    hovertemplate='%{x|%d %b}<br>Pronóstico: %{y:.1f} kWh<extra></extra>'))
fig_bars.add_trace(go.Bar(
    x=fechas_v, y=_val['Gen_GBM'],
    name='GBM+Clima', marker_color='#FF8F00', opacity=0.8,
    hovertemplate='%{x|%d %b}<br>GBM: %{y:.1f} kWh<extra></extra>'))

fig_bars.update_layout(
    title=f'<b>Real vs Pronóstico vs GBM — Mayo 18–24 2026</b><br>'
          f'<span style="font-size:11px;color:#555">'
          f'MAPE Pronóstico: {_mape_pron:.1f}%  |  MAPE GBM: {_mape_gbm:.1f}%</span>',
    barmode='group',
    xaxis=dict(title='Fecha', tickformat='%a %d %b', showgrid=True, gridcolor='#eee'),
    yaxis=dict(title='kWh/día', rangemode='tozero', showgrid=True, gridcolor='#eee'),
    plot_bgcolor='white', height=430,
    legend=dict(orientation='h', y=-0.2))
fig_bars.show()

# Resumen de ganadores por día
gbm_wins  = (mejor_vals[:-1].count('GBM'))
pron_wins = (mejor_vals[:-1].count('Pronóstico'))
print(f'\n=== Resumen 18-24 mayo 2026 ===')
print(f'{"Fecha":<16} {"Real":>7} {"Pronóst":>8} {"GBM":>8} {"ΔPron":>8} {"ΔGBM":>8} {"Mejor"}')
print('-'*70)
for _, r in _val.iterrows():
    print(f"{r['Fecha'].strftime('%a %d %b'):<16} {r['Real_kWh']:>7.1f} {r['Pron_kWh']:>8.1f} "
          f"{r['Gen_GBM']:>8.1f} {r['Err_Pron']:>+8.1f} {r['Err_GBM']:>+8.1f}  {r['Mejor']}")
print('-'*70)
print(f"{'TOTAL':<16} {_totales['Real_kWh']:>7.1f} {_totales['Pron_kWh']:>8.1f} "
      f"{_totales['Gen_GBM']:>8.1f} {_totales['Err_Pron']:>+8.1f} {_totales['Err_GBM']:>+8.1f}  {_totales['Mejor']}")
print()
print(f'GBM ganó {gbm_wins}/7 días | Pronóstico oficial ganó {pron_wins}/7 días')
print(f'MAPE Pronóstico: {_mape_pron:.1f}%  |  MAPE GBM+Clima: {_mape_gbm:.1f}%')



=== Resumen 18-24 mayo 2026 ===
Fecha               Real  Pronóst      GBM    ΔPron     ΔGBM Mejor
----------------------------------------------------------------------
Mon 18 May         227.1    285.6    217.2    +58.5     -9.9  GBM
Tue 19 May         231.1    215.1    228.4    -16.0     -2.7  GBM
Wed 20 May         246.1    245.1    234.9     -1.0    -11.2  Pronóstico
Thu 21 May         196.2    211.6    246.7    +15.3    +50.5  Pronóstico
Fri 22 May         241.7    227.1    244.9    -14.5     +3.3  GBM
Sat 23 May         199.9    277.6    201.9    +77.7     +2.1  GBM
Sun 24 May         244.4    281.5    244.6    +37.1     +0.2  GBM
----------------------------------------------------------------------
TOTAL             1586.5   1743.6   1618.7   +157.1    +32.2  GBM

GBM ganó 5/7 días | Pronóstico oficial ganó 2/7 días
MAPE Pronóstico: 14.4%  |  MAPE GBM+Clima: 5.5%
